In [1]:
import numpy as np

假设数据的维度全部都是(B, C, H, W)

别忘记分数线下面要有 $\epsilon$

### 一、BatchNorm

缺点：
1. batch_size 太小统计量不准、不稳定
2. 训练和推理统计量不一样

无法适应变长的序列

公式:

$$ x^{*} = \alpha \cdot \frac{x - \mu}{\sqrt{(\sigma^{2} + \epsilon)}} + \beta $$

In [2]:
class BatchNorm:
    def __init__(self, alpha, beta):
        # 缩放系数
        self.alpha = alpha
        # 偏移系数
        self.beta = beta

    def forward(self, X):
        "X : (B, C, H, W)"

        # 沿着 Batch_size, Height, Weight 来计算平均数和方差
        X_mean = np.mean(X, axis=(0,2,3), keepdims=True)
        X_var = np.var(X, axis=(0,2,3), keepdims=True)
        # (1, C, 1, 1)
        print(X_mean.shape)

        return self.alpha * (X - X_mean) / np.sqrt(X_var + 1e-6) + self.beta
        

###### 单元测试

In [3]:
X = np.array([[[[1, 2], 
       [3, 4]], 
      [[5, 6], 
       [7, 8]]],
     [[[1, 2], 
       [3, 4]],
      [[5, 6],
       [7, 8]]]])

# 会输出(2, 2, 2, 2)
print(X.shape)

BN = BatchNorm(alpha=0.9, beta=0)

# 会输出(1，2, 1, 1) 和一个其他的数组
BN.forward(X)

(2, 2, 2, 2)
(1, 2, 1, 1)


array([[[[-1.20747622, -0.40249207],
         [ 0.40249207,  1.20747622]],

        [[-1.20747622, -0.40249207],
         [ 0.40249207,  1.20747622]]],


       [[[-1.20747622, -0.40249207],
         [ 0.40249207,  1.20747622]],

        [[-1.20747622, -0.40249207],
         [ 0.40249207,  1.20747622]]]])

### 二、LayerNorm 

摆脱了对于 batch_size 的依赖

公式：

$$ x^{*} = \alpha \cdot \frac{x - \mu}{\sqrt{(\sigma^{2} + \epsilon)}} + \beta $$

In [4]:
# 优点在于不受批次数量的影响

class LayerNorm:
    def __init__(self, alpha, beta):
        # 缩放系数
        self.alpha = alpha
        # 偏移系数
        self.beta = beta

    def forward(self, X):
        "X : (B, C, H, W)"

        # 沿着 Channels, Height, Weight 来计算平均数和方差
        X_mean = np.mean(X, axis=(1,2,3), keepdims=True)
        X_var = np.var(X, axis=(1,2,3), keepdims=True)
        # (B, 1, 1, 1)
        print(X_mean.shape)

        return self.alpha * (X - X_mean) / np.sqrt(X_var + 1e-6) + self.beta

In [5]:
X = np.array([[[[1, 2], 
       [3, 4]], 
      [[5, 6], 
       [7, 8]]],
     [[[1, 2], 
       [3, 4]],
      [[5, 6],
       [7, 8]]]])

# 会输出(2, 2, 2, 2)
print(X.shape)

LN = LayerNorm(alpha=0.9, beta=0)

# 会输出(2，1, 1, 1) 和一个其他的数组
LN.forward(X)

(2, 2, 2, 2)
(2, 1, 1, 1)


array([[[[-1.37477258, -0.98198041],
         [-0.58918825, -0.19639608]],

        [[ 0.19639608,  0.58918825],
         [ 0.98198041,  1.37477258]]],


       [[[-1.37477258, -0.98198041],
         [-0.58918825, -0.19639608]],

        [[ 0.19639608,  0.58918825],
         [ 0.98198041,  1.37477258]]]])

### 三、InstanceNorm

公式：

$$ x^{*} = \alpha \cdot \frac{x - \mu}{\sqrt{(\sigma^{2} + \epsilon)}} + \beta $$

In [6]:
class InstanceNorm:
    def __init__(self, alpha, beta):
        # 缩放系数
        self.alpha = alpha
        # 偏移系数
        self.beta = beta

    def forward(self, X):
        "X : (B, C, H, W)"

        # 沿着 Height, Weight 来计算平均数和方差
        X_mean = np.mean(X, axis=(2,3), keepdims=True)
        X_var = np.var(X, axis=(2,3), keepdims=True)
        # (B, C, 1, 1)
        print(X_mean.shape)

        return self.alpha * (X - X_mean) / np.sqrt(X_var + 1e-6) + self.beta

In [7]:
X = np.array([[[[1, 2], 
       [3, 4]], 
      [[5, 6], 
       [7, 8]]],
     [[[1, 2], 
       [3, 4]],
      [[5, 6],
       [7, 8]]]])

# 会输出(2, 2, 2, 2)
print(X.shape)

IN = InstanceNorm(alpha=0.9, beta=0)

# 会输出(2，2, 1, 1) 和一个其他的数组
IN.forward(X)

(2, 2, 2, 2)
(2, 2, 1, 1)


array([[[[-1.20747622, -0.40249207],
         [ 0.40249207,  1.20747622]],

        [[-1.20747622, -0.40249207],
         [ 0.40249207,  1.20747622]]],


       [[[-1.20747622, -0.40249207],
         [ 0.40249207,  1.20747622]],

        [[-1.20747622, -0.40249207],
         [ 0.40249207,  1.20747622]]]])

### 四、GroupNorm

把通道数减少从而将小批量数据变为大批量数据

必须保证选择的组数能让 Channels 整除

公式：

$$ x^{*} = \alpha \cdot \frac{x - \mu}{\sqrt{(\sigma^{2} + \epsilon)}} + \beta $$

In [8]:
# 解决 BatchNorm 中的 Batch_size 太小的问题

class GroupNorm:
    def __init__(self, alpha, beta,group):
        # 缩放系数
        self.alpha = alpha
        # 偏移系数
        self.beta = beta

        self.group = group

    def forward(self, X):
        "X : (B, C, H, W)"

        X = X.reshape(X.shape[0] * self.group, X.shape[1] // self.group, X.shape[2], X.shape[3])
        print(X)

        # 沿着 Channels / Group, Height, Weight 来计算平均数和方差
        X_mean = np.mean(X, axis=(0,2,3), keepdims=True)
        X_var = np.var(X, axis=(0,2,3), keepdims=True)
        # (B, C, 1, 1)
        print(X_mean.shape)

        return self.alpha * (X - X_mean) / np.sqrt(X_var + 1e-6) + self.beta

In [9]:
X = np.array([[[[1, 2], 
       [3, 4]], 
      [[5, 6], 
       [7, 8]]],
     [[[1, 2], 
       [3, 4]],
      [[5, 6],
       [7, 8]]]])

# 会输出(2, 2, 2, 2)
print(X.shape)

GN = GroupNorm(alpha=0.9, beta=0, group=2)

# 会输出(1，1, 1, 1) 和一个其他的数组
GN.forward(X)

(2, 2, 2, 2)
[[[[1 2]
   [3 4]]]


 [[[5 6]
   [7 8]]]


 [[[1 2]
   [3 4]]]


 [[[5 6]
   [7 8]]]]
(1, 1, 1, 1)


array([[[[-1.37477258, -0.98198041],
         [-0.58918825, -0.19639608]]],


       [[[ 0.19639608,  0.58918825],
         [ 0.98198041,  1.37477258]]],


       [[[-1.37477258, -0.98198041],
         [-0.58918825, -0.19639608]]],


       [[[ 0.19639608,  0.58918825],
         [ 0.98198041,  1.37477258]]]])

### 五、RMSNorm

没有去中心化的操作，对于异常值更加鲁棒

提高了训练的效率

广泛应用于大模型

公式：

$$ x^* = \alpha \cdot \frac{x}{\sqrt{\sum\limits_{i=0}^{B}{x_{i}^{2}} + \epsilon}} $$

In [10]:
class RMSNorm:
    def __init__(self, alpha, beta):
        # 缩放系数
        self.alpha = alpha
        # 偏移系数
        self.beta = beta

    def forward(self, X):
        "X : (B, C, H, W)"

        X_square = np.sum(X ** 2)
        print(X_square)

        return self.alpha * X / np.sqrt(X_square /  + 1e-6) + self.beta

In [11]:
X = np.array([[[[1, 2], 
       [3, 4]], 
      [[5, 6], 
       [7, 8]]],
     [[[1, 2], 
       [3, 4]],
      [[5, 6],
       [7, 8]]]])

# 会输出(2, 2, 2, 2)
print(X.shape)

RMSN = RMSNorm(alpha=0.9, beta=0)

RMSN.forward(X)

(2, 2, 2, 2)
408


array([[[[4.45566394e-05, 8.91132789e-05],
         [1.33669918e-04, 1.78226558e-04]],

        [[2.22783197e-04, 2.67339837e-04],
         [3.11896476e-04, 3.56453115e-04]]],


       [[[4.45566394e-05, 8.91132789e-05],
         [1.33669918e-04, 1.78226558e-04]],

        [[2.22783197e-04, 2.67339837e-04],
         [3.11896476e-04, 3.56453115e-04]]]])